<a href="https://www.kaggle.com/code/shravankumarpandey/digit-recognizer-using-cnn?scriptVersionId=329383771" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.utils import to_categorical

# ====================================================
# Reproducibility
# ====================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ====================================================
# Load Dataset
# ====================================================

train_df = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/test.csv"
)

# ====================================================
# Prepare Data
# ====================================================

X = train_df.drop("label", axis=1).values
y = train_df["label"].values

X_test = test_df.values

X = X.astype("float32") / 255.
X_test = X_test.astype("float32") / 255.

X = X.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

y = to_categorical(y,10)

# ====================================================
# Train Validation Split
# ====================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=SEED,
    stratify=np.argmax(y,axis=1)
)

# ====================================================
# Data Augmentation
# ====================================================

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)

datagen.fit(X_train)

# ====================================================
# Build Model
# ====================================================

model = Sequential()

model.add(
    Conv2D(
        32,
        (3,3),
        padding="same",
        activation="relu",
        input_shape=(28,28,1)
    )
)
model.add(BatchNormalization())

model.add(
    Conv2D(
        32,
        (3,3),
        padding="same",
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(MaxPooling2D())
model.add(Dropout(0.25))

model.add(
    Conv2D(
        64,
        (3,3),
        padding="same",
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(
    Conv2D(
        64,
        (3,3),
        padding="same",
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(MaxPooling2D())
model.add(Dropout(0.25))

model.add(
    Conv2D(
        128,
        (3,3),
        padding="same",
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(
    Conv2D(
        128,
        (3,3),
        padding="same",
        activation="relu"
    )
)
model.add(BatchNormalization())

model.add(MaxPooling2D())
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(512,activation="relu"))
model.add(Dropout(0.50))

model.add(Dense(10,activation="softmax"))

# ====================================================
# Compile
# ====================================================

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=0.001,
    weight_decay=1e-4
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.05
    ),
    metrics=["accuracy"]
)

# ====================================================
# Callbacks
# ====================================================

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_accuracy",
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

# ====================================================
# Train
# ====================================================

history = model.fit(
    datagen.flow(
        X_train,
        y_train,
        batch_size=32
    ),
    epochs=50,
    validation_data=(X_val,y_val),
    callbacks=[
        early_stop,
        reduce_lr
    ],
    verbose=1
)

# ====================================================
# Validation Accuracy
# ====================================================

loss, acc = model.evaluate(
    X_val,
    y_val,
    verbose=0
)

print("\nValidation Accuracy:",acc)

# ====================================================
# Predict
# ====================================================

predictions = model.predict(
    X_test,
    verbose=1
)

predicted_labels = np.argmax(
    predictions,
    axis=1
)

# ====================================================
# Submission
# ====================================================

submission = pd.read_csv(
    "/kaggle/input/competitions/digit-recognizer/sample_submission.csv"
)

submission["Label"] = predicted_labels

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

print("\nSubmission Created!")

print("\nFiles:")
print(os.listdir("/kaggle/working"))

2026-06-22 05:18:16.917530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782105497.134257      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782105497.198052      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782105497.706452      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782105497.706499      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782105497.706503      22 computation_placer.cc:177] computation placer alr

Epoch 1/50


I0000 00:00:1782105522.142802      68 service.cc:152] XLA service 0x7fe2dc00d5f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782105522.142835      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782105522.142838      68 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782105523.019526      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


  16/1182 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.1474 - loss: 4.6794

I0000 00:00:1782105529.635582      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1182/1182 ━━━━━━━━━━━━━━━━━━━━ 33s 18ms/step - accuracy: 0.8866 - loss: 0.6779 - val_accuracy: 0.9836 - val_loss: 0.3811 - learning_rate: 0.0010
Epoch 2/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9690 - loss: 0.4360 - val_accuracy: 0.9881 - val_loss: 0.3432 - learning_rate: 0.0010
Epoch 3/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.9773 - loss: 0.3985 - val_accuracy: 0.9907 - val_loss: 0.3354 - learning_rate: 0.0010
Epoch 4/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9819 - loss: 0.3798 - val_accuracy: 0.9914 - val_loss: 0.3235 - learning_rate: 0.0010
Epoch 5/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9840 - loss: 0.3691 - val_accuracy: 0.9931 - val_loss: 0.3182 - learning_rate: 0.0010
Epoch 6/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9861 - loss: 0.3563 - val_accuracy: 0.9936 - val_loss: 0.3131 - learning_rate: 0.0010
Epoch 7/50
1182/1182 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9879 